# 3b · REVEAL Depth Sweep
---

Sweep all available depth slices in the REVEAL seismic tomography NetCDF to identify
which S-wave (Vsv) and P-wave (Vpv) depths best predict heat flow.

**Three stages:**
1. **Univariate CV R²** — each depth slice against heat flow in isolation
2. **Vp/Vs ratio & diff tests** — evaluate ratio pairs and gradient features constructed on-the-fly
3. **Informative figures** — depth profiles, correlation maps, feature distributions

**Output:**
- `output/depth_sweep/depth_sweep_results.csv` — ranked slices
- `output/depth_sweep/vpvs_test_results.csv` — Vp/Vs derived features
- `fig/depth_sweep/` — all figures
- A ready-to-paste `obs_list` update for `1_IMPORT.ipynb`

**Prerequisites:** `REVEAL.nc` exists at `../data/reveal/REVEAL.nc`


## 1 · Imports & Setup

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score
from quantile_forest import RandomForestQuantileRegressor
from scipy.stats import pearsonr

from config import *

# ── Output directories ────────────────────────────────────────────────────────
sweep_dir = output_root / 'depth_sweep'
fig_dir   = fig_root  / 'depth_sweep'
sweep_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# ── REVEAL NetCDF path ────────────────────────────────────────────────────────
REVEAL_NC = data_root / 'reveal' / 'REVEAL.nc'

print(f'REVEAL NetCDF : {REVEAL_NC}')
print(f'Exists        : {REVEAL_NC.exists()}')
print(f'Sweep output  : {sweep_dir}')
print(f'Figure output : {fig_dir}')


✓ config.py v3.0 | obs_model: 22 | obs_sweep: 34 | obs: 43
REVEAL NetCDF : ../data/reveal/REVEAL.nc
Exists        : True
Sweep output  : output/depth_sweep
Figure output : fig/depth_sweep


## 2 · Explore REVEAL NetCDF

Inspect available depths and variables before sweeping.

In [2]:
ds = xr.open_dataset(REVEAL_NC)
print(ds)
print()
print('── Variables ─────────────────────────────────────')
for v in ds.data_vars:
    print(f'  {v:20s}  {ds[v].dims}  shape={ds[v].shape}')
print()
print('── Coordinates ───────────────────────────────────')
for c in ds.coords:
    arr = ds[c].values
    print(f'  {c:20s}  n={len(arr)}  range=[{arr.min():.1f}, {arr.max():.1f}]')


<xarray.Dataset> Size: 412MB
Dimensions:    (depth: 99, latitude: 361, longitude: 721)
Coordinates:
  * depth      (depth) float32 396B 0.0 10.0 20.0 ... 2.8e+03 2.85e+03 2.88e+03
  * latitude   (latitude) float32 1kB -90.0 -89.5 -89.0 -88.5 ... 89.0 89.5 90.0
  * longitude  (longitude) float32 3kB -180.0 -179.5 -179.0 ... 179.5 180.0
Data variables:
    vsv        (depth, latitude, longitude) float32 103MB ...
    vsh        (depth, latitude, longitude) float32 103MB ...
    vpv        (depth, latitude, longitude) float32 103MB ...
    rho        (depth, latitude, longitude) float32 103MB ...
Attributes: (12/32)
    title:                         REVEAL: A Global Full‐Waveform Inversion M...
    id:                            REVEAL
    reference:                     REVEAL: A Global Full‐Waveform Inversion M...
    reference_pid:                 doi:10.1785/0120230273
    summary:                       The REVEAL model
    keywords:                      Full-Waveform Inversion, 1D st

### Available depth levels

In [3]:
depths_all = ds['depth'].values
print(f'All depths (km): {depths_all.tolist()}')

max_depth_km = 400

ds = ds.sel(depth=ds['depth'] <= max_depth_km)

depths_all = ds['depth'].values
print(f'Depths above {max_depth_km} (km): {depths_all.tolist()}')

# Identify S-wave and P-wave depth arrays
# vsv = S-wave (SV),  vpv = P-wave (PV)
S_DEPTHS = depths_all.tolist()   # sweep all depths for both
P_DEPTHS = depths_all.tolist()

print(f'\nS-wave depths to sweep : {S_DEPTHS}')
print(f'P-wave depths to sweep : {P_DEPTHS}')

All depths (km): [0.0, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0, 100.0, 110.0, 120.0, 130.0, 140.0, 150.0, 160.0, 170.0, 180.0, 190.0, 200.0, 210.0, 220.0, 230.0, 240.0, 250.0, 260.0, 270.0, 280.0, 290.0, 300.0, 310.0, 320.0, 330.0, 340.0, 350.0, 360.0, 370.0, 380.0, 390.0, 400.0, 410.0, 420.0, 430.0, 440.0, 450.0, 460.0, 470.0, 480.0, 490.0, 500.0, 550.0, 600.0, 650.0, 700.0, 750.0, 800.0, 850.0, 900.0, 950.0, 1000.0, 1050.0, 1100.0, 1150.0, 1200.0, 1250.0, 1300.0, 1350.0, 1400.0, 1450.0, 1500.0, 1550.0, 1600.0, 1650.0, 1700.0, 1750.0, 1800.0, 1850.0, 1900.0, 1950.0, 2000.0, 2050.0, 2100.0, 2150.0, 2200.0, 2250.0, 2300.0, 2350.0, 2400.0, 2450.0, 2500.0, 2550.0, 2600.0, 2650.0, 2700.0, 2750.0, 2800.0, 2850.0, 2880.0]
Depths above 400 (km): [0.0, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0, 100.0, 110.0, 120.0, 130.0, 140.0, 150.0, 160.0, 170.0, 180.0, 190.0, 200.0, 210.0, 220.0, 230.0, 240.0, 250.0, 260.0, 270.0, 280.0, 290.0, 300.0, 310.0, 320.0, 330.0, 340.0, 350

## 3 · Load Reference Heat Flow Data

In [4]:
ref = pd.read_parquet(parquet_ref).copy()
print(f'Reference observations : {len(ref):,}')
print(f'q range                : {ref["q"].min()*1000:.1f}–{ref["q"].max()*1000:.1f} mW/m²')

lons = ref['lon'].values
lats = ref['lat'].values
q    = ref['q'].values


Reference observations : 30,848
q range                : 1.0–350.0 mW/m²


## 4 · Sample REVEAL at Reference Points

Interpolate each depth slice directly from the NetCDF to the reference point locations.
This is the same approach used in `1_IMPORT`, ensuring the sweep tests the actual imported values.


In [5]:
def sample_reveal_at_points(ds, var, depth_km, lons, lats):
    """Bilinear interpolation of one REVEAL depth slice at (lon, lat) points."""
    da = ds[var].sel(depth=depth_km, method='nearest')
    lon_da = xr.DataArray(lons, dims='points')
    lat_da = xr.DataArray(lats, dims='points')
    vals = da.interp(longitude=lon_da, latitude=lat_da, method='linear').values
    return vals.astype(np.float64)

# Build a DataFrame of all depth slices sampled at reference locations
print('Sampling REVEAL slices at reference points...')
sampled = {}

for d in S_DEPTHS:
    key = f'REVEAL_S{int(d)}'
    try:
        sampled[key] = sample_reveal_at_points(ds, 'vsv', d, lons, lats)
        print(f'  {key:20s}  valid={np.isfinite(sampled[key]).sum():,}')
    except Exception as e:
        print(f'  {key:20s}  ERROR: {e}')

for d in P_DEPTHS:
    key = f'REVEAL_P{int(d)}'
    try:
        sampled[key] = sample_reveal_at_points(ds, 'vpv', d, lons, lats)
        print(f'  {key:20s}  valid={np.isfinite(sampled[key]).sum():,}')
    except Exception as e:
        print(f'  {key:20s}  ERROR: {e}')

reveal_df = pd.DataFrame(sampled)
reveal_df['q'] = q
reveal_df['lon'] = lons
reveal_df['lat'] = lats
print(f'\nSampled DataFrame: {reveal_df.shape}')


Sampling REVEAL slices at reference points...
  REVEAL_S0             valid=30,848
  REVEAL_S10            valid=30,848
  REVEAL_S20            valid=30,848
  REVEAL_S30            valid=30,848
  REVEAL_S40            valid=30,848
  REVEAL_S50            valid=30,848
  REVEAL_S60            valid=30,848
  REVEAL_S70            valid=30,848
  REVEAL_S80            valid=30,848
  REVEAL_S90            valid=30,848
  REVEAL_S100           valid=30,848
  REVEAL_S110           valid=30,848
  REVEAL_S120           valid=30,848
  REVEAL_S130           valid=30,848
  REVEAL_S140           valid=30,848
  REVEAL_S150           valid=30,848
  REVEAL_S160           valid=30,848
  REVEAL_S170           valid=30,848
  REVEAL_S180           valid=30,848
  REVEAL_S190           valid=30,848
  REVEAL_S200           valid=30,848
  REVEAL_S210           valid=30,848
  REVEAL_S220           valid=30,848
  REVEAL_S230           valid=30,848
  REVEAL_S240           valid=30,848
  REVEAL_S250           valid

## 5 · Univariate Depth Sweep

Train a lightweight QRF on each depth slice individually (5-fold CV).
Toggle `RUN_DEPTH_SWEEP` to skip if already saved.


In [6]:
RUN_DEPTH_SWEEP = True

depth_csv = sweep_dir / 'depth_sweep_results.csv'

if RUN_DEPTH_SWEEP:
    slice_cols = [c for c in reveal_df.columns if c.startswith('REVEAL_')]
    kf = KFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=random_state)

    rows = []
    for feat in slice_cols:
        sub = reveal_df[['q', feat]].dropna()
        X1  = sub[[feat]].values
        y1  = sub['q'].values

        qrf1 = RandomForestQuantileRegressor(
            n_estimators=500, max_features=1.0,
            min_samples_leaf=QRF_MIN_SAMPLES_LEAF,
            max_depth=QRF_MAX_DEPTH, n_jobs=QRF_N_JOBS,
            random_state=random_state,
        )
        r2 = cross_val_score(qrf1, X1, y1, cv=kf, scoring='r2').mean()
        r, p = pearsonr(X1[:, 0], y1)

        # Parse wave type and depth
        wave = 'S' if '_S' in feat else 'P'
        depth_km = int(feat.split(wave)[-1])

        rows.append({
            'feature'          : feat,
            'wave_type'        : wave,
            'depth_km'         : depth_km,
            'univariate_cv_r2' : r2,
            'pearson_r'        : r,
            'n_valid'          : len(sub),
        })
        print(f'  {feat:20s}  R²={r2:.4f}  r={r:+.3f}  n={len(sub):,}')

    depth_df = pd.DataFrame(rows).sort_values('univariate_cv_r2', ascending=False)
    depth_df.to_csv(depth_csv, index=False)
    print(f'\n✓ Saved: {depth_csv}')

else:
    depth_df = pd.read_csv(depth_csv).sort_values('univariate_cv_r2', ascending=False)
    print(f'Loaded: {depth_csv}')

print('\n── Top 10 depth slices ───────────────────────────────')
print(depth_df.head(10).to_string(index=False, float_format=lambda x: f'{x:.4f}'))


  REVEAL_S0             R²=-0.1232  r=-0.068  n=30,848
  REVEAL_S10            R²=-0.1323  r=-0.000  n=30,848
  REVEAL_S20            R²=-0.1538  r=+0.082  n=30,848
  REVEAL_S30            R²=-0.1287  r=+0.034  n=30,848
  REVEAL_S40            R²=-0.0916  r=-0.165  n=30,848
  REVEAL_S50            R²=-0.0774  r=-0.324  n=30,848
  REVEAL_S60            R²=-0.0331  r=-0.373  n=30,848
  REVEAL_S70            R²=0.0090  r=-0.394  n=30,848
  REVEAL_S80            R²=0.0379  r=-0.394  n=30,848
  REVEAL_S90            R²=0.0143  r=-0.387  n=30,848
  REVEAL_S100           R²=0.0056  r=-0.367  n=30,848
  REVEAL_S110           R²=-0.0187  r=-0.342  n=30,848
  REVEAL_S120           R²=-0.0126  r=-0.317  n=30,848
  REVEAL_S130           R²=-0.0621  r=-0.294  n=30,848
  REVEAL_S140           R²=-0.0575  r=-0.273  n=30,848
  REVEAL_S150           R²=-0.1069  r=-0.255  n=30,848
  REVEAL_S160           R²=-0.0897  r=-0.240  n=30,848
  REVEAL_S170           R²=-0.1039  r=-0.225  n=30,848
  REVEAL_S180 

## 6 · Vp/Vs Ratio & Gradient Tests

Construct candidate Vp/Vs ratio and velocity-gradient features from the sampled slices,
then test each with the same univariate CV approach.

Candidates follow the same logic as the existing `obs_list`:
- **Vp/Vs ratio**: Vpv(d1) / Vsv(d2) where depths are close (same lithospheric window)
- **S-gradient**: Vsv(d2) − Vsv(d1)  (velocity change with depth)
- **P-gradient**: Vpv(d2) − Vpv(d1)


In [ ]:
RUN_VPVS_SWEEP = True

vpvs_csv = sweep_dir / 'vpvs_test_results.csv'

if RUN_VPVS_SWEEP:
    s_feats = [c for c in reveal_df.columns if c.startswith('REVEAL_S')]
    p_feats = [c for c in reveal_df.columns if c.startswith('REVEAL_P')]
    s_depths = sorted([int(f.split('S')[-1]) for f in s_feats])
    p_depths = sorted([int(f.split('P')[-1]) for f in p_feats])

    kf = KFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=random_state)
    rows = []

    def test_derived(label, values, q_vals, wave_type='derived'):
        sub_mask = np.isfinite(values)
        X1 = values[sub_mask].reshape(-1, 1)
        y1 = q_vals[sub_mask]
        if len(X1) < 100:
            return
        qrf1 = RandomForestQuantileRegressor(
            n_estimators=500, max_features=1.0,
            min_samples_leaf=QRF_MIN_SAMPLES_LEAF,
            max_depth=QRF_MAX_DEPTH, n_jobs=QRF_N_JOBS,
            random_state=random_state,
        )
        r2 = cross_val_score(qrf1, X1, y1, cv=kf, scoring='r2').mean()
        r, _ = pearsonr(X1[:, 0], y1)
        print(f'  {label:25s}  R²={r2:.4f}  r={r:+.3f}  n={sub_mask.sum():,}')
        rows.append({'feature': label, 'wave_type': wave_type,
                     'univariate_cv_r2': r2, 'pearson_r': r, 'n_valid': sub_mask.sum()})

    q_arr = reveal_df['q'].values

    print('── Vp/Vs ratios ──────────────────────────────────────────')
    for pd_ in p_depths:
        for sd_ in s_depths:
            if abs(pd_ - sd_) <= 30:    # only test paired depths within 30 km
                vp = reveal_df.get(f'REVEAL_P{pd_}')
                vs = reveal_df.get(f'REVEAL_S{sd_}')
                if vp is not None and vs is not None:
                    with np.errstate(divide='ignore', invalid='ignore'):
                        ratio = np.where(vs != 0, vp.values / vs.values, np.nan)
                    test_derived(f'VP{pd_}VS{sd_}', ratio, q_arr, 'VpVs')

    print('\n── S-wave gradients ──────────────────────────────────────')
    for i in range(len(s_depths)):
        for j in range(i+1, len(s_depths)):
            d1, d2 = s_depths[i], s_depths[j]
            if d2 - d1 <= 60:   # test adjacent pairs only
                vs1 = reveal_df[f'REVEAL_S{d1}'].values
                vs2 = reveal_df[f'REVEAL_S{d2}'].values
                test_derived(f'SDIFF{d1}{d2}', vs2 - vs1, q_arr, 'S-grad')

    print('\n── P-wave gradients ──────────────────────────────────────')
    for i in range(len(p_depths)):
        for j in range(i+1, len(p_depths)):
            d1, d2 = p_depths[i], p_depths[j]
            if d2 - d1 <= 80:
                vp1 = reveal_df[f'REVEAL_P{d1}'].values
                vp2 = reveal_df[f'REVEAL_P{d2}'].values
                test_derived(f'PDIFF{d1}{d2}', vp2 - vp1, q_arr, 'P-grad')

    print('\n── Global Vp/Vs difference ───────────────────────────────')
    # VPVS_DIFF: mean(Vpv) / mean(Vsv) across all depths
    mean_vs = np.nanmean(reveal_df[[f'REVEAL_S{d}' for d in s_depths]].values, axis=1)
    mean_vp = np.nanmean(reveal_df[[f'REVEAL_P{d}' for d in p_depths]].values, axis=1)
    with np.errstate(divide='ignore', invalid='ignore'):
        vpvs_diff = np.where(mean_vs != 0, mean_vp / mean_vs, np.nan)
    test_derived('VPVS_DIFF', vpvs_diff, q_arr, 'VpVs-global')

    vpvs_df = pd.DataFrame(rows).sort_values('univariate_cv_r2', ascending=False)
    vpvs_df.to_csv(vpvs_csv, index=False)
    print(f'\n✓ Saved: {vpvs_csv}')

else:
    vpvs_df = pd.read_csv(vpvs_csv).sort_values('univariate_cv_r2', ascending=False)
    print(f'Loaded: {vpvs_csv}')

print('\n── Top Vp/Vs & gradient features ─────────────────────────')
print(vpvs_df.head(15).to_string(index=False, float_format=lambda x: f'{x:.4f}'))


✓ config.py v4.0 | obs_model: 23 | obs_sweep: 35 | obs: 44
── Vp/Vs ratios ──────────────────────────────────────────
  VP0VS0                     R²=-0.1152  r=+0.047  n=30,848
  VP0VS10                    R²=-0.1572  r=-0.028  n=30,848
  VP0VS20                    R²=-0.1018  r=-0.104  n=30,848
  VP0VS30                    R²=-0.1196  r=-0.059  n=30,848
  VP10VS0                    R²=-0.1528  r=+0.070  n=30,848
  VP10VS10                   R²=-0.1030  r=+0.070  n=30,848


## 7 · Figures

### 7.1 CV R² vs depth — S-wave and P-wave

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=False)

for ax, wave, color in zip(axes, ['S', 'P'], ['steelblue', 'firebrick']):
    sub = depth_df[depth_df['wave_type'] == wave].sort_values('depth_km')
    ax.plot(sub['univariate_cv_r2'], sub['depth_km'], 'o-', color=color, lw=1.5, ms=6)
    for _, row in sub.iterrows():
        ax.annotate(f'{row["univariate_cv_r2"]:.3f}',
                    xy=(row['univariate_cv_r2'], row['depth_km']),
                    xytext=(4, 0), textcoords='offset points', fontsize=7)
    ax.set_xlabel('5-fold CV R²', fontsize=10)
    ax.set_ylabel('Depth (km)', fontsize=10)
    ax.set_title(f'{"S-wave (Vsv)" if wave=="S" else "P-wave (Vpv)"}', fontsize=11)
    ax.invert_yaxis()
    ax.axvline(0, color='k', lw=0.6, ls='--')
    ax.grid(True, alpha=0.3)

fig.suptitle('REVEAL univariate CV R² vs depth', fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(fig_dir / 'depth_r2_profile.png', dpi=fig_dpi, bbox_inches='tight')
plt.show()
print('Saved: depth_r2_profile.png')


### 7.2 Pearson correlation vs depth

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=False)

for ax, wave, color in zip(axes, ['S', 'P'], ['steelblue', 'firebrick']):
    sub = depth_df[depth_df['wave_type'] == wave].sort_values('depth_km')
    ax.barh(sub['depth_km'].astype(str), sub['pearson_r'], color=color, alpha=0.7)
    ax.axvline(0, color='k', lw=0.8)
    ax.set_xlabel('Pearson r (vs heat flow)', fontsize=10)
    ax.set_ylabel('Depth (km)', fontsize=10)
    ax.set_title(f'{"S-wave (Vsv)" if wave=="S" else "P-wave (Vpv)"}', fontsize=11)
    ax.grid(True, axis='x', alpha=0.3)
    ax.invert_yaxis()

fig.suptitle('REVEAL Pearson correlation with heat flow vs depth', fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(fig_dir / 'depth_pearson_r.png', dpi=fig_dpi, bbox_inches='tight')
plt.show()
print('Saved: depth_pearson_r.png')


### 7.3 Depth-averaged velocity vs heat flow (scatter, best slices)

In [ ]:
# Select top-3 S-wave and top-3 P-wave slices by CV R²
top_s = depth_df[depth_df['wave_type'] == 'S'].nlargest(3, 'univariate_cv_r2')
top_p = depth_df[depth_df['wave_type'] == 'P'].nlargest(3, 'univariate_cv_r2')
best_feats = pd.concat([top_s, top_p])

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes = axes.flatten()

for ax, (_, row) in zip(axes, best_feats.iterrows()):
    feat = row['feature']
    sub  = reveal_df[['q', feat, 'lon', 'lat']].dropna()
    x    = sub[feat].values
    y    = sub['q'].values * 1000   # → mW/m²
    ax.hexbin(x, y, gridsize=50, cmap='YlOrRd', mincnt=1)
    ax.set_xlabel(f'{feat} (km/s)', fontsize=8)
    ax.set_ylabel('q (mW/m²)', fontsize=8)
    ax.set_title(f'{feat}  R²={row["univariate_cv_r2"]:.3f}', fontsize=9)
    ax.tick_params(labelsize=7)

fig.suptitle('Velocity vs heat flow — top REVEAL slices', fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig(fig_dir / 'velocity_vs_hf_scatter.png', dpi=fig_dpi, bbox_inches='tight')
plt.show()
print('Saved: velocity_vs_hf_scatter.png')


### 7.4 Vp/Vs ratio & gradient ranking

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# Vp/Vs ratios
vpvs_only = vpvs_df[vpvs_df['wave_type'] == 'VpVs'].sort_values('univariate_cv_r2')
colors_vv = ['#d73027' if r < 0 else '#4575b4' for r in vpvs_only['pearson_r']]
axes[0].barh(vpvs_only['feature'], vpvs_only['univariate_cv_r2'], color=colors_vv, alpha=0.8)
axes[0].set_xlabel('5-fold CV R²', fontsize=10)
axes[0].set_title('Vp/Vs ratio features', fontsize=11)
axes[0].axvline(0, color='k', lw=0.6)
axes[0].grid(True, axis='x', alpha=0.3)

# Gradients
grad_only = vpvs_df[vpvs_df['wave_type'].isin(['S-grad', 'P-grad'])].sort_values('univariate_cv_r2')
col_map = {'S-grad': 'steelblue', 'P-grad': 'firebrick'}
colors_gr = [col_map[w] for w in grad_only['wave_type']]
axes[1].barh(grad_only['feature'], grad_only['univariate_cv_r2'], color=colors_gr, alpha=0.8)
axes[1].set_xlabel('5-fold CV R²', fontsize=10)
axes[1].set_title('Velocity gradient features  (blue=S, red=P)', fontsize=11)
axes[1].axvline(0, color='k', lw=0.6)
axes[1].grid(True, axis='x', alpha=0.3)

fig.suptitle('Derived REVEAL features — univariate CV R²', fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig(fig_dir / 'vpvs_gradient_ranking.png', dpi=fig_dpi, bbox_inches='tight')
plt.show()
print('Saved: vpvs_gradient_ranking.png')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ── Vp/Vs: scatter with Vp-depth on x, Vs-depth on y ─────────────────────────
vv = vpvs_df[vpvs_df['wave_type'] == 'VpVs'].copy()
# Parse depth pairs from feature name e.g. VP60VS70 → vp_d=60, vs_d=70
vv['vp_d'] = vv['feature'].str.extract(r'VP(\d+)VS').astype(int)
vv['vs_d'] = vv['feature'].str.extract(r'VS(\d+)').astype(int)

sc0 = axes[0].scatter(
    vv['vp_d'], vv['vs_d'],
    c=vv['univariate_cv_r2'],
    cmap='RdYlGn', vmin=-0.05, vmax=0.025,
    s=60, edgecolors='k', linewidths=0.3
)
axes[0].set_xlabel('Vpv depth (km)', fontsize=10)
axes[0].set_ylabel('Vsv depth (km)', fontsize=10)
axes[0].set_title('Vp/Vs ratio — univariate CV R²', fontsize=11)
plt.colorbar(sc0, ax=axes[0], label='CV R²', shrink=0.8)
axes[0].grid(True, alpha=0.2)

# ── Gradients: scatter with d1 on x, d2 on y, shape=wave type ────────────────
sg = vpvs_df[vpvs_df['wave_type'] == 'S-grad'].copy()
pg = vpvs_df[vpvs_df['wave_type'] == 'P-grad'].copy()

def parse_diff(df, prefix):
    nums = df['feature'].str.extract(rf'{prefix}DIFF(\d+)(\d{{2,3}})')
    # feature names like SDIFF80100 → d1=80, d2=100
    # re-parse robustly: strip prefix, split at the second depth boundary
    d1, d2 = [], []
    for f in df['feature']:
        body = f.replace(prefix + 'DIFF', '')
        # try splitting at every position
        for k in range(1, len(body)):
            a, b = body[:k], body[k:]
            if a and b and int(a) < int(b):
                d1.append(int(a)); d2.append(int(b))
                break
    df = df.iloc[:len(d1)].copy()
    df['d1'] = d1
    df['d2'] = d2
    return df

sg = parse_diff(sg, 'S')
pg = parse_diff(pg, 'P')

sc1s = axes[1].scatter(sg['d1'], sg['d2']/km, c=sg['univariate_cv_r2'],
                        cmap='RdYlGn', vmin=-0.12, vmax=0.0,
                        s=70, marker='o', edgecolors='steelblue',
                        linewidths=0.6, label='S-grad', alpha=0.8)
plt.colorbar(sc1s, ax=axes[1], label='CV R²', shrink=0.8)
axes[1].set_xlabel('Shallow depth d1 (km)', fontsize=10)
axes[1].set_ylabel('Deep depth d2 (km)', fontsize=10)
axes[1].set_title('Velocity gradients — univariate CV R²\n○ S-grad  □ P-grad', fontsize=11)
axes[1].legend(fontsize=9, loc='upper left')
axes[1].grid(True, alpha=0.2)



sc1s = axes[2].scatter(pg['d1'], pg['d2']/km, c=pg['univariate_cv_r2'],
                cmap='RdYlGn', vmin=-0.12, vmax=0.0,
                s=70, marker='s', edgecolors='firebrick',
                linewidths=0.6, label='P-grad', alpha=0.8)
plt.colorbar(sc1s, ax=axes[2], label='CV R²', shrink=0.8)
axes[2].set_xlabel('Shallow depth d1 (km)', fontsize=10)
axes[2].set_ylabel('Deep depth d2 (km)', fontsize=10)
axes[2].set_title('Velocity gradients — univariate CV R²\n○ S-grad  □ P-grad', fontsize=11)
axes[2].legend(fontsize=9, loc='upper left')
axes[2].grid(True, alpha=0.2)


fig.suptitle('Derived REVEAL features — univariate CV R²', fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig(fig_dir / 'vpvs_gradient_ranking.png', dpi=fig_dpi, bbox_inches='tight')
plt.show()
print('Saved: vpvs_gradient_ranking.png')

### 7.5 Global summary heatmap

In [ ]:
# Build a combined pivot: rows=depth, cols=S/P, value=CV R²
pivot_s = depth_df[depth_df['wave_type'] == 'S'].set_index('depth_km')['univariate_cv_r2']
pivot_p = depth_df[depth_df['wave_type'] == 'P'].set_index('depth_km')['univariate_cv_r2']

all_depths = sorted(set(pivot_s.index) | set(pivot_p.index))
matrix = pd.DataFrame({
    'Vsv (S)': [pivot_s.get(d, np.nan) for d in all_depths],
    'Vpv (P)': [pivot_p.get(d, np.nan) for d in all_depths],
}, index=all_depths)

fig, ax = plt.subplots(figsize=(6, max(4, len(all_depths) * 0.45)))
im = ax.imshow(matrix.values, aspect='auto', cmap='RdYlGn', vmin=-0.05, vmax=0.30)
ax.set_xticks([0, 1])
ax.set_xticklabels(matrix.columns, fontsize=10)
ax.set_yticks(range(len(all_depths)))
ax.set_yticklabels(all_depths, fontsize=9)
ax.set_ylabel('Depth (km)')
ax.set_title('Univariate CV R² heatmap\nREVEAL depth slices', fontsize=11)
plt.colorbar(im, ax=ax, label='CV R²', shrink=0.7)
# Annotate cells
for i, row in enumerate(matrix.values):
    for j, val in enumerate(row):
        if np.isfinite(val):
            ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=8,
                    color='white' if val > 0.15 else 'black')
fig.tight_layout()
fig.savefig(fig_dir / 'depth_r2_heatmap.png', dpi=fig_dpi, bbox_inches='tight')
plt.show()
print('Saved: depth_r2_heatmap.png')


## 8 · Optimal Selection & Ready-to-Paste `obs_list`

Selects the best N S-wave and M P-wave depths plus the best Vp/Vs and gradient
features.  Adjust the thresholds below before copying.


In [ ]:
# ── Thresholds — adjust as needed ────────────────────────────────────────────
N_S_SLICES = 3       # number of top S-wave depths to include
N_P_SLICES = 3       # number of top P-wave depths to include
N_VPVS     = 2       # top Vp/Vs ratios
N_GRADS    = 2       # top gradient features
R2_MIN     = 0.01    # minimum CV R² to include any feature

# ── Pick best slices ──────────────────────────────────────────────────────────
best_s = depth_df[depth_df['wave_type'] == 'S'].nlargest(N_S_SLICES, 'univariate_cv_r2')
best_s = best_s[best_s['univariate_cv_r2'] >= R2_MIN]

best_p = depth_df[depth_df['wave_type'] == 'P'].nlargest(N_P_SLICES, 'univariate_cv_r2')
best_p = best_p[best_p['univariate_cv_r2'] >= R2_MIN]

best_vv = vpvs_df[vpvs_df['wave_type'] == 'VpVs'].nlargest(N_VPVS, 'univariate_cv_r2')
best_vv = best_vv[best_vv['univariate_cv_r2'] >= R2_MIN]

best_gr = vpvs_df[vpvs_df['wave_type'].isin(['S-grad', 'P-grad'])].nlargest(N_GRADS, 'univariate_cv_r2')
best_gr = best_gr[best_gr['univariate_cv_r2'] >= R2_MIN]

print('── Selected depth slices ────────────────────────────────────────────')
print('S-wave:')
print(best_s[['feature', 'depth_km', 'univariate_cv_r2']].to_string(index=False))
print('\nP-wave:')
print(best_p[['feature', 'depth_km', 'univariate_cv_r2']].to_string(index=False))
print('\nVp/Vs ratios:')
print(best_vv[['feature', 'univariate_cv_r2']].to_string(index=False))
print('\nGradients:')
print(best_gr[['feature', 'univariate_cv_r2']].to_string(index=False))

# ── Build obs_list snippet ────────────────────────────────────────────────────
selected = (
    list(best_s['feature']) +
    list(best_p['feature']) +
    list(best_vv['feature']) +
    list(best_gr['feature'])
)

print('\n' + '─'*60)
print('# ── Paste these REVEAL entries into obs_list in config.py ──')
for f in selected:
    row = pd.concat([depth_df, vpvs_df]).set_index('feature').loc[f]
    print(f"    '{f}',  # CV R²={row['univariate_cv_r2']:.4f}")
print('─'*60)


## 9 · Inter-depth Correlation Matrix

Checks redundancy between depth slices before finalising the selection.

In [ ]:
# Use only the slices that have enough valid data
slice_cols = [c for c in reveal_df.columns if c.startswith('REVEAL_')]
corr_df = reveal_df[slice_cols].corr()

fig, ax = plt.subplots(figsize=(max(6, len(slice_cols)*0.7), max(5, len(slice_cols)*0.7)))
im = ax.imshow(corr_df.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_df)))
ax.set_xticklabels(corr_df.columns, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(len(corr_df)))
ax.set_yticklabels(corr_df.index, fontsize=7)
ax.set_title('Pearson correlation between REVEAL depth slices', fontsize=11)
plt.colorbar(im, ax=ax, shrink=0.7, label='r')
for i in range(len(corr_df)):
    for j in range(len(corr_df)):
        v = corr_df.values[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=5,
                color='white' if abs(v) > 0.7 else 'black')
fig.tight_layout()
fig.savefig(fig_dir / 'depth_correlation_matrix.png', dpi=fig_dpi, bbox_inches='tight')
plt.show()
print('Saved: depth_correlation_matrix.png')


## 10 · (Optional) Compile PDF

Run this cell after the figures are saved if you have a corresponding `.tex` file.

In [ ]:
import subprocess
tex_path = output_root.parent / 'tex' / 'depth_sweep'
if tex_path.exists():
    result = subprocess.run(
        ['pdflatex', '-interaction=nonstopmode', 'depth_sweep.tex'],
        cwd=tex_path, capture_output=True, text=True
    )
    print(result.stdout[-2000:] if result.stdout else '')
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])
else:
    print(f'No tex dir at {tex_path} — skipping PDF compile.')
